# 01 — Lists and Tuples

Two sequence types. **List = mutable, growable.** **Tuple = immutable, fixed.**

We finish with Python's *reference model* — the single biggest source of "wait, why did that change?" surprises.

## Lists — the workhorse

In [1]:
fruits = ["apple", "banana", "cherry"]
print(len(fruits))
print(fruits[0], fruits[-1])      # index from front and back
fruits.append("date")              # mutates in place
fruits.insert(0, "avocado")        # insert at index
print(fruits)
print("banana" in fruits)          # membership
removed = fruits.pop()             # remove & return last
print(removed, fruits)

3
apple cherry
['avocado', 'apple', 'banana', 'cherry', 'date']
True
date ['avocado', 'apple', 'banana', 'cherry']


### Slicing — `seq[start:stop:step]`

All three parts are optional. `stop` is **exclusive**. Negative indices count from the end.

Slicing returns a **new** list — it does *not* mutate the original.

In [2]:
nums = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
print(nums[2:5])      # [2, 3, 4]            indices 2,3,4
print(nums[:3])       # [0, 1, 2]            from start
print(nums[7:])       # [7, 8, 9]            to end
print(nums[-3:])      # [7, 8, 9]            last 3
print(nums[::2])      # [0, 2, 4, 6, 8]      every 2nd
print(nums[::-1])     # [9, 8, ..., 0]       reversed copy
print(nums)           # unchanged

[2, 3, 4]
[0, 1, 2]
[7, 8, 9]
[7, 8, 9]
[0, 2, 4, 6, 8]
[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


### Useful list operations

In [3]:
xs = [3, 1, 4, 1, 5, 9, 2, 6]

# Sorted COPY vs in-place sort
print(sorted(xs))             # returns a new list
print(xs)                     # unchanged
xs.sort()                     # mutates xs
print(xs)

# Sort by a key — descending by absolute value, for example
words = ["apple", "fig", "banana", "kiwi"]
print(sorted(words, key=len))               # by length
print(sorted(words, key=len, reverse=True)) # longest first

# Concatenation creates a new list
print([1, 2] + [3, 4])
print([0] * 5)                # repeat — handy, but watch the trap at the bottom

[1, 1, 2, 3, 4, 5, 6, 9]
[3, 1, 4, 1, 5, 9, 2, 6]
[1, 1, 2, 3, 4, 5, 6, 9]
['fig', 'kiwi', 'apple', 'banana']
['banana', 'apple', 'kiwi', 'fig']
[1, 2, 3, 4]
[0, 0, 0, 0, 0]


## Tuples — immutable, hashable, often used as records

Tuples are written with `()` (or no brackets at all in some contexts). Use a tuple when the
*shape* of the data is fixed and meaningful — coordinates, RGB colors, a row from a query.

In [4]:
point = (3, 4)              # tuple of two ints
x, y = point                # tuple unpacking
print(x, y)

single = (42,)              # one-element tuple needs the trailing comma
not_a_tuple = (42)          # this is just int 42 in parens
print(type(single), type(not_a_tuple))

# Tuples are immutable:
try:
    point[0] = 99
except TypeError as e:
    print("can't mutate:", e)

# Because they're immutable, they can be dict keys / set members:
edges = {("a", "b"), ("b", "c")}
print(edges)

3 4
<class 'tuple'> <class 'int'>
can't mutate: 'tuple' object does not support item assignment
{('a', 'b'), ('b', 'c')}


### Swap and multiple return values

Tuple unpacking is how Python does multi-return and pythonic swap.

In [5]:
a, b = 1, 2
a, b = b, a                 # swap — no temp needed
print(a, b)

def min_max(xs):
    return min(xs), max(xs)  # returns a tuple

lo, hi = min_max([5, 2, 9, 1, 7])
print(lo, hi)

2 1
1 9


## Mutability — the mental model

Some types are **mutable**: `list`, `dict`, `set`. You can change them after creation.
Some are **immutable**: `int`, `float`, `str`, `tuple`, `bool`, `None`, `frozenset`. You cannot.

"Mutating" means the *same object* changes. "Rebinding" means the name now points to a *different* object.

In [6]:
xs = [1, 2, 3]
print(id(xs))         # id() returns the object's identity (in CPython, its memory address)
xs.append(4)          # MUTATION — same object
print(id(xs), xs)     # same id

xs = xs + [5]         # REBINDING — new list created, name points to it
print(id(xs), xs)     # different id

2707652236480
2707652236480 [1, 2, 3, 4]
2707652238336 [1, 2, 3, 4, 5]


## References, not values — the classic trap

`b = a` does **not** copy. Both names point to the same object. If the object is mutable,
changes through one name are visible through the other.

In [7]:
a = [1, 2, 3]
b = a                 # b and a refer to the SAME list
b.append(99)
print("a:", a)        # [1, 2, 3, 99]   — surprising if you didn't know
print("b:", b)
print("same object?", a is b)

a: [1, 2, 3, 99]
b: [1, 2, 3, 99]
same object? True


### Copying

- `list(a)` or `a[:]` or `a.copy()` make a **shallow** copy: a new outer list, but inner objects are shared.
- `copy.deepcopy(a)` recurses, copying everything.

Shallow is usually enough. Deep matters when your structure has nested mutables.

In [8]:
import copy

a = [[1, 2], [3, 4]]
shallow = a.copy()
deep    = copy.deepcopy(a)

a[0].append(99)        # mutate an INNER list of `a`
print("a      :", a)
print("shallow:", shallow)   # also sees 99 — inner list is shared
print("deep   :", deep)      # untouched

a      : [[1, 2, 99], [3, 4]]
shallow: [[1, 2, 99], [3, 4]]
deep   : [[1, 2], [3, 4]]


## The `[0] * n` trap (revisited)

`[0] * 3` is fine — `int` is immutable. But `[[0] * 3] * 3` creates *three references to the same inner list*.

In [9]:
bad  = [[0] * 3] * 3            # three refs to ONE inner list
good = [[0] * 3 for _ in range(3)]  # comprehension creates a fresh inner list each time

bad[0][0] = 9
good[0][0] = 9
print("bad :", bad)
print("good:", good)

bad : [[9, 0, 0], [9, 0, 0], [9, 0, 0]]
good: [[9, 0, 0], [0, 0, 0], [0, 0, 0]]


## Mini-exercise

1. Given `xs = [3, 1, 4, 1, 5, 9, 2, 6]`, produce **a new list** with duplicates removed *while preserving order* — without using a set comprehension. (Hint: track "seen" with a `set`.)
2. Predict, then run: what does `nums[::-1]` give you for an empty list?
3. Why is `a is b` not the same as `a == b`? Construct an example where they disagree in both directions.

In [10]:
# your solutions here


**Takeaways**

- `list` is mutable; `tuple` is immutable and hashable.
- Slicing returns a *new* list. `seq.sort()` mutates; `sorted(seq)` doesn't.
- `b = a` aliases — it does not copy.
- `*` repetition on a list of mutables makes shared references. Use a comprehension to get fresh ones.

Next: [02_dicts_and_sets.ipynb](02_dicts_and_sets.ipynb)